# 01. R4T 기초: 집합 수준 검색 지표

원문: **Efficient, Property-Aligned Fan-Out Retrieval via RL-Compiled Diffusion** (arXiv:2603.06397v1, 2026-03-06)

> 이 노트북은 논문의 핵심 수식을 작은 합성 임베딩으로 확인하는 **toy reproduction**입니다. Polyvore·음악 데이터, 저자 모델, LLM judge 및 실제 RL 학습을 재현하지 않으며 논문 성능을 재현했다고 주장하지 않습니다. 네트워크·API·GPU를 사용하지 않습니다.

## 학습 목표

1. 점 하나가 아니라 결과 **집합**을 평가해야 하는 이유를 이해합니다.
2. 논문의 alignment, groundedness, Vendi diversity, coverage를 NumPy로 계산합니다.
3. 서로 충돌할 수 있는 보상 항목을 가중합할 때 생기는 스케일 문제를 확인합니다.

실행 환경: Python 3, NumPy. 위에서 아래 순서로 실행하세요. 모든 난수는 고정 seed를 사용합니다.

In [1]:
import numpy as np

SEED = 260306397
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

def normalize(x, eps=1e-12):
    # 마지막 축을 단위 길이로 바꿉니다. 코사인 유사도를 내적으로 계산하기 위한 전처리입니다.
    x = np.asarray(x, dtype=float)
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

print(f'NumPy {np.__version__} / deterministic seed={SEED}')

NumPy 2.5.1 / deterministic seed=260306397


## 논문의 네 가지 지표

광범위 질의 `q`를 `k`개 하위 질의의 집합 `Q={q_i}`로 펼친다고 합시다. 논문 식 (1)~(5)의 핵심은 다음과 같습니다.

- **정렬도(alignment)**: `mean_i cos(e_text(q_i), e_text(q))`
- **근거성(groundedness)**: `1 - mean_i min_c ||e_text(q_i)-e_content(c)||_2`
- **다양성(diversity)**: 각 하위 질의의 대표 검색 결과에 대한 Vendi Score
- **커버리지(coverage)**: `|Y intersection R(Q)| / |Y|`

OAR(open-ended abstract retrieval)는 정답 집합이 없으므로 앞의 세 속성을 복합 보상으로 사용합니다. WSCR(weakly supervised compositional retrieval)은 하나의 가능한 참조 집합 `Y`를 이용해 coverage를 계산합니다. 참조 집합은 유일한 정답이 아닙니다.

In [2]:
# 네 차원은 실제 의미 임베딩을 흉내 낸 좌표일 뿐입니다.
item_names = np.array([
    'boho_dress', 'festival_boots', 'woven_bag', 'minimal_jacket',
    'formal_suit', 'sport_shoe', 'vintage_hat', 'beach_sandal'
])
database = normalize(np.array([
    [0.90,  0.30,  0.15, 0.05],
    [0.80,  0.10,  0.50, 0.05],
    [0.75,  0.45,  0.10, 0.05],
    [0.70, -0.15, -0.20, 0.10],
    [0.50, -0.40, -0.25, 0.20],
    [0.35, -0.10,  0.75, 0.10],
    [0.65,  0.45, -0.10, 0.20],
    [0.55,  0.20,  0.55, 0.10],
]))

broad_query = normalize(np.array([1.0, 0.10, 0.10, 0.05]))
fanout_good = normalize(np.array([
    [0.90,  0.30,  0.15, 0.05],
    [0.80,  0.10,  0.50, 0.05],
    [0.75,  0.45,  0.10, 0.05],
    [0.70, -0.15, -0.20, 0.10],
]))
fanout_repeated = np.repeat(fanout_good[:1], repeats=4, axis=0)
reference = {'boho_dress', 'festival_boots', 'woven_bag', 'minimal_jacket'}

print('database:', database.shape, '/ fan-out:', fanout_good.shape)

database: (8, 4) / fan-out: (4, 4)


In [3]:
def nearest_ids(queries, contents, top_k=1):
    # 단위 벡터이므로 행렬곱이 코사인 유사도입니다. 한 번의 배치 연산으로 검색합니다.
    similarities = normalize(queries) @ normalize(contents).T
    return np.argsort(-similarities, axis=1)[:, :top_k]

def vendi_score(embeddings, eps=1e-12):
    # Vendi Score = exp(Shannon entropy of normalized kernel eigenvalues).
    # 코사인 Gram 행렬은 단위 벡터의 내적이므로 positive semidefinite입니다.
    x = normalize(embeddings)
    kernel = x @ x.T
    eigenvalues = np.linalg.eigvalsh(kernel)
    probabilities = np.clip(eigenvalues, 0.0, None)
    probabilities /= probabilities.sum()
    entropy = -np.sum(probabilities * np.log(probabilities + eps))
    return float(np.exp(entropy))

def evaluate_set(query, fanout, contents, names, reference_set):
    ids = nearest_ids(fanout, contents, top_k=1).ravel()
    representatives = contents[ids]
    alignment = float(np.mean(normalize(fanout) @ normalize(query)))
    nearest_distance = np.linalg.norm(normalize(fanout) - representatives, axis=1)
    groundedness = float(1.0 - nearest_distance.mean())
    diversity = vendi_score(representatives)
    retrieved = set(names[ids].tolist())
    coverage = len(reference_set & retrieved) / len(reference_set)
    # 논문의 OAR 기본 가중치: lambda_g=0.6, lambda_d=lambda_a=0.2
    composite = 0.6 * groundedness + 0.2 * diversity + 0.2 * alignment
    return {
        'alignment': alignment, 'groundedness': groundedness,
        'vendi': diversity, 'coverage': coverage, 'composite': composite,
        'retrieved': sorted(retrieved),
    }

good = evaluate_set(broad_query, fanout_good, database, item_names, reference)
repeated = evaluate_set(broad_query, fanout_repeated, database, item_names, reference)

In [4]:
for label, result in [('diverse fan-out', good), ('repeated fan-out', repeated)]:
    numeric = {k: round(v, 3) for k, v in result.items() if isinstance(v, float)}
    print(f'{label:>20}: {numeric}')
    print(' ' * 22 + f"retrieved={result['retrieved']}")

# 핵심 성질을 자동 검증합니다.
assert 0.0 <= good['alignment'] <= 1.0
assert good['groundedness'] > 0.99  # 이 toy에서는 하위 질의가 DB 항목과 정확히 일치합니다.
assert 1.0 <= good['vendi'] <= len(fanout_good) + 1e-9
assert np.isclose(repeated['vendi'], 1.0, atol=1e-8)
assert good['vendi'] > repeated['vendi']
assert good['coverage'] == 1.0 and repeated['coverage'] == 0.25
assert good['composite'] > repeated['composite']
print('Assertions passed: 집합 수준 지표가 중복 fan-out의 한계를 감지했습니다.')

     diverse fan-out: {'alignment': 0.915, 'groundedness': 1.0, 'vendi': 1.663, 'coverage': 1.0, 'composite': 1.116}
                      retrieved=['boho_dress', 'festival_boots', 'minimal_jacket', 'woven_bag']
    repeated fan-out: {'alignment': 0.974, 'groundedness': 1.0, 'vendi': 1.0, 'coverage': 0.25, 'composite': 0.995}
                      retrieved=['boho_dress']
Assertions passed: 집합 수준 지표가 중복 fan-out의 한계를 감지했습니다.


## 결과 읽기와 예상 출력

`diverse fan-out`은 네 참조 항목을 모두 찾고 Vendi Score가 1보다 큽니다. `repeated fan-out`은 alignment가 높더라도 같은 항목만 반복하므로 Vendi Score는 1, coverage는 0.25입니다. 마지막 셀은 `Assertions passed`로 끝나야 합니다.

## 주의점과 한계

- Vendi Score의 범위는 대략 1부터 집합 크기까지인 반면 다른 보상은 대체로 0~1입니다. 실제 학습에서는 정규화와 가중치 보정이 중요합니다.
- 논문의 groundedness 식은 임베딩 거리 스케일에 민감하고 이론적으로 음수가 될 수 있습니다. 이 예제는 단위 벡터를 사용했습니다.
- 논문 OAR 평가는 LLM-as-a-Judge도 사용합니다. 여기의 수치 지표는 그 평가를 대신하지 않습니다.
- WSCR coverage는 하나의 약한 참조 집합을 기준으로 한 proxy이며 검색 결과의 절대적 정답률이 아닙니다.
- 다음 노트북에서는 보상이 높은 fan-out 궤적을 선택해 순서가 없는 목표 텐서로 컴파일합니다.